# Retrain Models with Engineered Features

This notebook retrains the regression models using the updated engineered feature set.

In [1]:
from pathlib import Path

import pandas as pd

import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
DATA_PATH = Path(
    "../data/processed/crmls_sfr_engineered_full.parquet"
)

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (398461, 83)


,source_period,source_file,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,CloseDate,...,LivingAreaPerBathroom,LotToLivingRatio,LogLivingArea,LogLotSize,AmenityKnownCount,AmenityCount,CloseMonthSin,CloseMonthCos,split_month,UnifiedSchoolDistrict
0,2022-01,CRMLSSold20220101_20231231_filled.csv,None,True,None,None,False,2499999.0,556365765,2022-01-04,...,661.25,5.057089,7.880804,9.501292,4,2.0,0.5,0.866025,2022-01,Carlsbad Unified
1,2022-01,CRMLSSold20220101_20231231_filled.csv,None,True,None,None,False,565000.0,556365286,2022-01-10,...,690.00,1.641063,7.635787,8.130942,5,3.0,0.5,0.866025,2022-01,Rim of the World Unified
2,2022-01,CRMLSSold20220101_20231231_filled.csv,"Carpet,Tile",True,None,None,True,599000.0,556364261,2022-01-10,...,998.00,3.055110,7.599401,8.715880,4,3.0,0.5,0.866025,2022-01,Palm Springs Unified
3,2022-01,CRMLSSold20220101_20231231_filled.csv,None,True,None,None,False,399990.0,556363890,2022-01-19,...,711.00,8.577356,7.260523,9.409027,5,3.0,0.5,0.866025,2022-01,Paradise Unified
4,2022-01,CRMLSSold20220101_20231231_filled.csv,None,False,None,None,False,1849000.0,556362674,2022-01-03,...,943.25,2.006361,8.235891,8.932080,4,2.0,0.5,0.866025,2022-01,Poway Unified


In [3]:
df["CloseDate"] = pd.to_datetime(
    df["CloseDate"],
    errors="coerce"
)

df["split_month"] = (
    df["CloseDate"]
    .dt.to_period("M")
)

In [4]:
monthly_counts = (
    df.groupby("split_month")
    .size()
    .rename("RowCount")
)

monthly_counts.tail(12)

split_month
2025-06    11677
2025-07    12099
2025-08    11430
2025-09    11435
2025-10    12010
2025-11     9713
2025-12    10438
2026-01     7469
2026-02     8538
2026-03    11165
2026-04    12014
2026-05    12013
Freq: M, Name: RowCount, dtype: int64

Test and validation month

In [5]:
test_month = df["split_month"].max()

validation_month = test_month - 1

print("Validation month:", validation_month)
print("Test month:", test_month)

Validation month: 2026-04
Test month: 2026-05


In [6]:
train_df = df[
    df["split_month"] < validation_month
].copy()

validation_df = df[
    df["split_month"] == validation_month
].copy()

test_df = df[
    df["split_month"] == test_month
].copy()

In [7]:
print(
    "Train:",
    train_df.shape,
    train_df["split_month"].min(),
    "to",
    train_df["split_month"].max()
)

print(
    "Validation:",
    validation_df.shape,
    validation_df["split_month"].min(),
    "to",
    validation_df["split_month"].max()
)

print(
    "Test:",
    test_df.shape,
    test_df["split_month"].min(),
    "to",
    test_df["split_month"].max()
)

Train: (374434, 83) 2022-01 to 2026-03
Validation: (12014, 83) 2026-04 to 2026-04
Test: (12013, 83) 2026-05 to 2026-05


In [8]:
TARGET = "ClosePrice"

y_train = train_df[TARGET].copy()

y_validation = validation_df[TARGET].copy()

y_test = test_df[TARGET].copy()

## 2. Define the Updated Feature Set

The retrained models use the original baseline features together with the newly engineered property, geographic, amenity, and temporal features.

Only explicitly approved variables are selected. Identifier columns, transaction-related variables, pricing fields, and other leakage-prone variables are excluded from the model.

In [9]:
BASE_NUMERIC_FEATURES = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "ParkingTotal",
    "Stories"
]


ADDED_NUMERIC_FEATURES = [
    "Latitude",
    "Longitude"
]


ENGINEERED_NUMERIC_FEATURES = [
    "PropertyAge",
    "BathBedRatio",
    "LivingAreaPerBedroom",
    "LivingAreaPerBathroom",
    "LotToLivingRatio",
    "LogLivingArea",
    "LogLotSize",
    "AmenityCount",
    "AmenityKnownCount",
    "CloseMonthSin",
    "CloseMonthCos"
]


NUMERIC_FEATURES = (
    BASE_NUMERIC_FEATURES
    + ADDED_NUMERIC_FEATURES
    + ENGINEERED_NUMERIC_FEATURES
)

In [10]:
BASE_CATEGORICAL_FEATURES = [
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "Levels",
    "PoolPrivateYN",
    "ViewYN",
    "AttachedGarageYN",
    "NewConstructionYN",
    "FireplaceYN"
]


ADDED_CATEGORICAL_FEATURES = [
    "City",
    "UnifiedSchoolDistrict"
]


CATEGORICAL_FEATURES = (
    BASE_CATEGORICAL_FEATURES
    + ADDED_CATEGORICAL_FEATURES
)

In [11]:
MODEL_FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

print("Numeric feature count:", len(NUMERIC_FEATURES))
print("Categorical feature count:", len(CATEGORICAL_FEATURES))
print("Total raw feature count:", len(MODEL_FEATURES))

Numeric feature count: 21
Categorical feature count: 11
Total raw feature count: 32


## 3. Train-Only Target Outlier Treatment

ClosePrice outlier thresholds are calculated using only the training data.

The 0.5th and 99.5th percentiles from the training set are frozen and applied unchanged to the validation and test sets. This prevents validation or test-set distributions from influencing preprocessing decisions.

In [12]:
LOWER_QUANTILE = 0.005
UPPER_QUANTILE = 0.995


lower_price = train_df["ClosePrice"].quantile(
    LOWER_QUANTILE
)

upper_price = train_df["ClosePrice"].quantile(
    UPPER_QUANTILE
)


print(
    "Lower ClosePrice threshold:",
    f"${lower_price:,.0f}"
)

print(
    "Upper ClosePrice threshold:",
    f"${upper_price:,.0f}"
)

Lower ClosePrice threshold: $188,500
Upper ClosePrice threshold: $8,200,000


In [13]:
validation_df_unfiltered = validation_df.copy()
test_df_unfiltered = test_df.copy()

In [14]:
train_df = train_df[
    train_df["ClosePrice"].between(
        lower_price,
        upper_price
    )
].copy()


validation_df = validation_df[
    validation_df["ClosePrice"].between(
        lower_price,
        upper_price
    )
].copy()


test_df = test_df[
    test_df["ClosePrice"].between(
        lower_price,
        upper_price
    )
].copy()


print("Filtered train shape:", train_df.shape)
print("Filtered validation shape:", validation_df.shape)
print("Filtered test shape:", test_df.shape)

print(
    "Unfiltered validation shape:",
    validation_df_unfiltered.shape
)

print(
    "Unfiltered test shape:",
    test_df_unfiltered.shape
)

Filtered train shape: (370695, 83)
Filtered validation shape: (11882, 83)
Filtered test shape: (11890, 83)
Unfiltered validation shape: (12014, 83)
Unfiltered test shape: (12013, 83)


In [15]:
TARGET = "ClosePrice"


X_train = train_df[
    MODEL_FEATURES
].copy()

y_train = train_df[
    TARGET
].copy()


X_validation = validation_df[
    MODEL_FEATURES
].copy()

y_validation = validation_df[
    TARGET
].copy()


X_test = test_df[
    MODEL_FEATURES
].copy()

y_test = test_df[
    TARGET
].copy()

In [16]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (370695, 32)
y_train: (370695,)
X_validation: (11882, 32)
y_validation: (11882,)
X_test: (11890, 32)
y_test: (11890,)


Save Data

In [17]:
SAVE_DIR = Path(
    "../data/processed/model_splits"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [18]:
train_data = X_train.copy()
train_data[TARGET] = y_train.values

validation_data = X_validation.copy()
validation_data[TARGET] = y_validation.values

test_data = X_test.copy()
test_data[TARGET] = y_test.values

In [19]:
train_data.to_parquet(
    SAVE_DIR / "train_engineered.parquet",
    index=False
)

validation_data.to_parquet(
    SAVE_DIR / "validation_engineered.parquet",
    index=False
)

test_data.to_parquet(
    SAVE_DIR / "test_engineered.parquet",
    index=False
)

In [20]:
print("Train saved:", train_data.shape)
print("Validation saved:", validation_data.shape)
print("Test saved:", test_data.shape)

Train saved: (370695, 33)
Validation saved: (11882, 33)
Test saved: (11890, 33)


## 4. Build the Preprocessing Pipelines

Before model training, the selected features are prepared for machine learning.

In [21]:
X_train[NUMERIC_FEATURES] = (
    X_train[NUMERIC_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

X_validation[NUMERIC_FEATURES] = (
    X_validation[NUMERIC_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

X_test[NUMERIC_FEATURES] = (
    X_test[NUMERIC_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

In [22]:
print(
    "Train infinite values:",
    X_train[NUMERIC_FEATURES]
    .isin([np.inf, -np.inf])
    .sum()
    .sum()
)

print(
    "Validation infinite values:",
    X_validation[NUMERIC_FEATURES]
    .isin([np.inf, -np.inf])
    .sum()
    .sum()
)

print(
    "Test infinite values:",
    X_test[NUMERIC_FEATURES]
    .isin([np.inf, -np.inf])
    .sum()
    .sum()
)

Train infinite values: 0
Validation infinite values: 0
Test infinite values: 0


Categorical Pipeline

In [23]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=50
            )
        )
    ]
)

Linear Regression Numeric Pipeline

LR needed numerical scaling

In [24]:
linear_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            linear_numeric_pipeline,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)

Tree Model Numeric Pipeline

In [25]:
tree_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        )
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_pipeline,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)

## 5. Retrain the Linear Regression Baseline

The linear preprocessing steps and the Linear Regression model are combined into one Pipeline.

In [26]:
linear_model = Pipeline(
    steps=[
        (
            "preprocessor",
            linear_preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)

In [27]:
for data in [X_train, X_validation, X_test]:
    for feature in CATEGORICAL_FEATURES:
        data[feature] = (
            data[feature]
            .where(data[feature].notna(), "Unknown")
            .astype(str)
        )

In [28]:
linear_model.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
validation_predictions = linear_model.predict(
    X_validation
)

In [30]:
validation_r2 = r2_score(
    y_validation,
    validation_predictions
)

validation_mae = mean_absolute_error(
    y_validation,
    validation_predictions
)

validation_mape = mean_absolute_percentage_error(
    y_validation,
    validation_predictions
) * 100

validation_mdape = np.median(
    np.abs(
        (
            y_validation
            - validation_predictions
        )
        / y_validation
    )
) * 100


print(
    "Validation R²:",
    round(validation_r2, 4)
)

print(
    "Validation MAE:",
    f"${validation_mae:,.2f}"
)

print(
    "Validation MAPE:",
    f"{validation_mape:.2f}%"
)

print(
    "Validation MdAPE:",
    f"{validation_mdape:.2f}%"
)

Validation R²: 0.8322
Validation MAE: $233,462.50
Validation MAPE: 21.29%
Validation MdAPE: 14.52%


## 6. Train Tree-Based Models

Two tree-based regression models are trained using the same engineered feature set:

- Decision Tree Regressor
- Random Forest Regressor

The models use a preprocessing pipeline without numerical scaling because tree-based models do not require standardized numerical features.

Model performance is first evaluated on the validation set.

In [31]:
decision_tree_model = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=20,
                min_samples_leaf=10,
                random_state=42
            )
        )
    ]
)

In [32]:
decision_tree_model.fit(
    X_train,
    y_train
)

print("Decision Tree training completed.")

Decision Tree training completed.


In [33]:
decision_tree_predictions = decision_tree_model.predict(
    X_validation
)

In [34]:
decision_tree_r2 = r2_score(
    y_validation,
    decision_tree_predictions
)

decision_tree_mae = mean_absolute_error(
    y_validation,
    decision_tree_predictions
)

decision_tree_mape = mean_absolute_percentage_error(
    y_validation,
    decision_tree_predictions
) * 100

decision_tree_mdape = np.median(
    np.abs(
        (
            y_validation
            - decision_tree_predictions
        )
        / y_validation
    )
) * 100

print(
    "Decision Tree Validation R²:",
    round(decision_tree_r2, 4)
)

print(
    "Decision Tree Validation MAE:",
    f"${decision_tree_mae:,.2f}"
)

print(
    "Decision Tree Validation MAPE:",
    f"{decision_tree_mape:.2f}%"
)

print(
    "Decision Tree Validation MdAPE:",
    f"{decision_tree_mdape:.2f}%"
)

Decision Tree Validation R²: 0.8218
Decision Tree Validation MAE: $218,168.03
Decision Tree Validation MAPE: 16.94%
Decision Tree Validation MdAPE: 11.03%


Random Forest

In [35]:
random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [36]:
random_forest_model.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [37]:
random_forest_predictions = random_forest_model.predict(
    X_validation
)

In [38]:
random_forest_r2 = r2_score(
    y_validation,
    random_forest_predictions
)

random_forest_mae = mean_absolute_error(
    y_validation,
    random_forest_predictions
)

random_forest_mape = mean_absolute_percentage_error(
    y_validation,
    random_forest_predictions
) * 100

random_forest_mdape = np.median(
    np.abs(
        (
            y_validation
            - random_forest_predictions
        )
        / y_validation
    )
) * 100


print(
    "Random Forest Validation R²:",
    round(random_forest_r2, 4)
)

print(
    "Random Forest Validation MAE:",
    f"${random_forest_mae:,.2f}"
)

print(
    "Random Forest Validation MAPE:",
    f"{random_forest_mape:.2f}%"
)

print(
    "Random Forest Validation MdAPE:",
    f"{random_forest_mdape:.2f}%"
)

Random Forest Validation R²: 0.8763
Random Forest Validation MAE: $179,151.26
Random Forest Validation MAPE: 13.88%
Random Forest Validation MdAPE: 9.19%


## Result

In [39]:
model_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "R2": [
        validation_r2,
        decision_tree_r2,
        random_forest_r2
    ],
    "MAE": [
        validation_mae,
        decision_tree_mae,
        random_forest_mae
    ],
    "MAPE": [
        validation_mape,
        decision_tree_mape,
        random_forest_mape
    ],
    "MdAPE": [
        validation_mdape,
        decision_tree_mdape,
        random_forest_mdape
    ]
})

model_results

,Model,R2,MAE,MAPE,MdAPE
0,Linear Regression,0.832154,233462.496725,21.292336,14.522312
1,Decision Tree,0.821750,218168.028422,16.943511,11.034414
2,Random Forest,0.876305,179151.261127,13.876988,9.188410


Baseline Model Performance (Before Feature Engineering):

| Model | Test R2 |
| :--- | :---: |
| Linear Regression | 0.8326 |
| Random Forest | 0.7801 |
| Decision Tree | 0.7355 |

In [40]:
model_results.sort_values(
    by="MdAPE"
)

,Model,R2,MAE,MAPE,MdAPE
2,Random Forest,0.876305,179151.261127,13.876988,9.188410
1,Decision Tree,0.821750,218168.028422,16.943511,11.034414
0,Linear Regression,0.832154,233462.496725,21.292336,14.522312
